# 12 — Frozen Proof-period evaluation

**Project:** Hotel overbooking decision support  
**Stage:** Final retrospective evaluation on the reserved Proof period  
**Purpose:** Evaluate the model and policy selected using Train and Choose, without further tuning.

This notebook is the final holdout-style evaluation step:
1. Record the model, settings and policy selected in Notebook 11.
2. Fit the frozen model using Train only.
3. Generate predictions for Proof.
4. Apply the frozen policy without changing its settings.
5. Score it with the unchanged client contract.
6. Report group and hotel-level results.

**No tuning, recalibration, threshold changes, policy changes or model replacement are allowed after inspecting Proof results.**

### Historical disclosure

The Proof period has already been inspected in previous project work. This evaluation is therefore retrospective, not a historically blind test. It can still provide a disciplined final comparison if the model and policy are frozen before this notebook is run, but it must not be described as a genuinely unseen test.

In [1]:
# Run from the repository root, so `from src import ...` and the relative evidence/ paths work
# whether this notebook is opened from the repo folder or from notebooks/pipeline/.
import os
import sys
from pathlib import Path

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "contract.py").exists())
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("Working directory:", REPO_ROOT)

Working directory: C:\Users\Lenovo\Desktop\Build_2_Repo\IIT_GN_W06


## 1. Read the frozen decision before running

The cell below reads the configuration Notebook 11 froze in `evidence/choose_policy_selection/frozen_configuration.json`, and stops if the Notebook 08 or 09 evidence behind it has changed since.

(Edited 2026-11-18: this cell previously held hand-copied placeholders, logistic regression with `C=1.0` and `model_buffer_1.0`, which did not match what Notebooks 08 and 11 selected.) Nothing here is chosen by looking at Proof outcomes.

In [2]:
# Read the configuration frozen by Notebook 11 (this cell previously held hand-copied placeholders:
# logistic regression C=1.0 and model_buffer_1.0, which matched nothing actually selected).
import hashlib
import json
from pathlib import Path

FREEZE_PATH = Path("evidence/choose_policy_selection/frozen_configuration.json")
frozen = json.loads(FREEZE_PATH.read_text(encoding="utf-8"))

# The Notebook 08 and 09 evidence the freeze was built from must be unchanged since the freeze.
for path, digest in frozen["inputs_sha256"].items():
    assert hashlib.sha256(Path(path).read_bytes()).hexdigest() == digest, f"{path} changed after the freeze"

FROZEN_MODEL_NAME = frozen["model"]
FROZEN_MODEL_SETTINGS = frozen["model_settings"]
FROZEN_POLICY_NAME = frozen["selected_policy"]
assert FROZEN_POLICY_NAME is not None, "Notebook 11 found no policy meeting the walk ceiling; there is nothing to evaluate."
FROZEN_RULE = frozen["selected_policy_rule"]
FROZEN_BUFFER_MULTIPLIER = FROZEN_RULE.get("multiplier")
USE_CALIBRATED_PROBABILITIES = frozen["probability_method"] == "calibrated"
BENCHMARKS = {"flat rule (contract baseline)": frozen["flat_rule_benchmark"],
              "best model-based policy on Choose": frozen["model_policy_benchmark"]}

print("Frozen at:", frozen["frozen_at"])
print("Frozen model:", FROZEN_MODEL_NAME, FROZEN_MODEL_SETTINGS, "| probabilities:", frozen["probability_method"])
print("Frozen policy:", FROZEN_POLICY_NAME, FROZEN_RULE)
print("Declared benchmarks:", {k: (v or {}).get("policy") for k, v in BENCHMARKS.items()})
print("Disclosure:", frozen["historical_disclosure"])

Frozen at: 2026-11-18T02:32:44
Frozen model: logistic_regression {'C': 0.1} | probabilities: calibrated
Frozen policy: flat_rule_margin_17 {'family': 'flat_rule_margin', 'margin': 17}
Declared benchmarks: {'flat rule (contract baseline)': 'flat_rule_margin_17', 'best model-based policy on Choose': 'model_margin_18'}
Disclosure: The Proof period was inspected in earlier project work (see docs/DECISIONS.md, 2026-11-18). Notebook 12's evaluation is retrospective, not historically blind.


## 2. Load data and the unchanged contract

The supplied contract defines the Train, Choose and Proof boundaries and the scoring function. Do not edit `src/contract.py`.

In [3]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src import data, contract

raw = data.read_raw()
bookings = raw.copy()
bookings["arrival_date"] = data.arrival_date(bookings)
bookings["nights"] = (
    bookings["stays_in_weekend_nights"] + bookings["stays_in_week_nights"]
)

train, choose, proof = contract.split_periods(bookings)

print("Train rows:", f"{len(train):,}")
print("Choose rows:", f"{len(choose):,}")
print("Proof rows:", f"{len(proof):,}")

Train rows: 49,157
Choose rows: 29,546
Proof rows: 40,687


## 3. Verify the chronological boundaries

This checks that the rows being evaluated belong to the period defined as Proof by the contract. It does not make the historical data blind.

In [4]:
assert len(train) > 0 and len(choose) > 0 and len(proof) > 0
assert train["arrival_date"].max() < pd.Timestamp(contract.TRAIN_END)
assert choose["arrival_date"].min() >= pd.Timestamp(contract.TRAIN_END)
assert choose["arrival_date"].max() < pd.Timestamp(contract.PROOF_START)
assert proof["arrival_date"].min() >= pd.Timestamp(contract.PROOF_START)

print("Chronological boundaries match the supplied contract.")
print("Proof dates:", proof["arrival_date"].min().date(), "to",
      proof["arrival_date"].max().date())

Chronological boundaries match the supplied contract.
Proof dates: 2017-01-01 to 2017-08-31


## 4. Validate the frozen configuration

This cell prevents unsupported settings or policy names from silently being used.

In [5]:
allowed_models = {"logistic_regression", "random_forest"}
fixed_policies = {"no_overbooking", "global_rate", "hotel_rate", "hotel_month_lookup", "model_expected"}
parameterised = {"model_buffer": "multiplier", "model_margin": "margin", "flat_rule_margin": "margin"}

assert FROZEN_MODEL_NAME in allowed_models
for rule in [FROZEN_RULE, *[b for b in BENCHMARKS.values() if b]]:
    assert rule["family"] in fixed_policies or rule["family"] in parameterised, rule
    if rule["family"] in parameterised:
        assert parameterised[rule["family"]] in rule, rule

if FROZEN_MODEL_NAME == "logistic_regression":
    assert set(FROZEN_MODEL_SETTINGS) == {"C"}
elif FROZEN_MODEL_NAME == "random_forest":
    assert set(FROZEN_MODEL_SETTINGS) == {"n_estimators", "min_samples_leaf"}

if USE_CALIBRATED_PROBABILITIES:
    assert frozen["calibrator_intercept"] is not None and frozen["calibrator_coefficient"] is not None

print("Frozen configuration checks passed.")

Frozen configuration checks passed.


## 5. Prepare the same candidate features

This matches Notebooks 05–11. `country` remains held back because its availability at the original booking timestamp has not been verified.

In [6]:
categorical_columns = [
    "hotel",
    "market_segment",
    "distribution_channel",
    "agent",
    "company"
]

numeric_columns = [
    "is_repeated_guest",
    "previous_cancellations",
    "previous_bookings_not_canceled"
]

feature_columns = categorical_columns + numeric_columns

def prepare_features(frame):
    X = frame[feature_columns].copy()

    for column in categorical_columns:
        X[column] = X[column].astype("string")

    X["agent"] = X["agent"].fillna("NO_AGENT")
    X["company"] = X["company"].fillna("NO_COMPANY")

    for column in ["hotel", "market_segment", "distribution_channel"]:
        X[column] = X[column].fillna("UNKNOWN_" + column.upper())

    return X

## 6. Fit the frozen model on Train only

The preprocessor is fitted as part of the model pipeline. Proof is not passed to `.fit()`.

In [7]:
def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore"),
                categorical_columns
            ),
            (
                "numeric",
                Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler())
                ]),
                numeric_columns
            )
        ],
        remainder="drop"
    )

if FROZEN_MODEL_NAME == "logistic_regression":
    estimator = LogisticRegression(
        C=FROZEN_MODEL_SETTINGS["C"],
        max_iter=1000
    )
else:
    estimator = RandomForestClassifier(
        n_estimators=FROZEN_MODEL_SETTINGS["n_estimators"],
        min_samples_leaf=FROZEN_MODEL_SETTINGS["min_samples_leaf"],
        random_state=42,
        n_jobs=-1
    )

frozen_model = Pipeline(steps=[
    ("preprocess", make_preprocessor()),
    ("model", estimator)
])

X_train = prepare_features(train)
y_train = train["is_canceled"].astype(int)

frozen_model.fit(X_train, y_train)

print("Frozen model fitted on Train.")

Frozen model fitted on Train.


## 7. Generate cancellation probabilities for Proof

The positive class is `1`, meaning cancellation or no-show. These predictions are generated before comparing them with Proof outcomes.

In [8]:
X_proof = prepare_features(proof)

class_values = list(frozen_model.named_steps["model"].classes_)
positive_position = class_values.index(1)

proof_probability = frozen_model.predict_proba(
    X_proof
)[:, positive_position]

assert len(proof_probability) == len(proof)
assert np.isfinite(proof_probability).all()
assert np.all((proof_probability >= 0) & (proof_probability <= 1))

print("Proof booking probabilities created:", len(proof_probability))

Proof booking probabilities created: 40687


In [9]:
# Apply the frozen calibration mapping from Notebook 09, exactly as Notebook 11 did on Choose.
proof_probability_uncalibrated = proof_probability
if USE_CALIBRATED_PROBABILITIES:
    proof_probability = 1.0 / (1.0 + np.exp(-(frozen["calibrator_intercept"]
                                              + frozen["calibrator_coefficient"] * proof_probability_uncalibrated)))
assert np.all((proof_probability >= 0) & (proof_probability <= 1))
print("Probability method:", frozen["probability_method"])

Probability method: calibrated


## 8. Build the historical lookup estimates from Train

The frozen policy may be a historical baseline rather than a model-based rule. We calculate the global, hotel and hotel-month rates from Train only, using the same fallback logic as Notebook 11.

In [10]:
global_rate = train["is_canceled"].mean()
hotel_rates = train.groupby("hotel")["is_canceled"].mean()

train_month = train.assign(
    arrival_month=train["arrival_date"].dt.month
)
proof_month = proof.assign(
    arrival_month=proof["arrival_date"].dt.month
)

hotel_month_rates = train_month.groupby(
    ["hotel", "arrival_month"]
)["is_canceled"].mean()

proof_lookup = proof_month[["hotel", "arrival_month"]].copy()
proof_lookup["global_rate"] = global_rate
proof_lookup["hotel_rate"] = (
    proof_lookup["hotel"].map(hotel_rates).fillna(global_rate)
)

proof_keys = pd.MultiIndex.from_frame(
    proof_lookup[["hotel", "arrival_month"]]
)
proof_lookup["hotel_month_rate"] = hotel_month_rates.reindex(
    proof_keys
).to_numpy()

proof_lookup["hotel_month_rate"] = proof_lookup["hotel_month_rate"].fillna(
    proof_lookup["hotel_rate"]
)

## 9. Aggregate Proof bookings to hotel-arrival nights

We use the exact contract aggregation function. It validates that every booking has one probability and calculates the expected cancellation count and spread for each hotel-arrival night.

In [11]:
proof_scoring = proof[
    ["hotel", "arrival_date", "is_canceled", "adr", "nights"]
].copy()

proof_nights = contract.nightly_table(
    proof_scoring,
    proof_probability
)

proof_global_nights = contract.nightly_table(
    proof_scoring,
    np.full(len(proof), global_rate)
)

proof_hotel_probability = (
    proof["hotel"].map(hotel_rates).fillna(global_rate).to_numpy()
)
proof_hotel_nights = contract.nightly_table(
    proof_scoring,
    proof_hotel_probability
)

proof_lookup_nights = contract.nightly_table(
    proof_scoring,
    proof_lookup["hotel_month_rate"].to_numpy()
)

display(proof_nights.head())

book  cancelled   expected    spread  stay_value
hotel      arrival_date                                                  
City Hotel 2017-01-01     103         39  62.331381  4.301767  258.124369
           2017-01-02     124         48  45.474639  5.288324  359.438548
           2017-01-03      95         36  31.411214  4.473005  330.703263
           2017-01-04      61         24  19.267803  3.528850  282.513607
           2017-01-05      65         34  26.900409  3.751709  335.478769

## 10. Apply the frozen policy without changing it

The selected policy name determines the exact rule. We do not search for a better buffer or choose a different policy after observing Proof outcomes.

In [12]:
def oversell_for(rule):
    """Integer oversell count per Proof hotel-night for one frozen rule (same rounding as Notebook 11)."""
    family = rule["family"]
    if family == "no_overbooking":
        counts = np.zeros(len(proof_nights), dtype=int)
    elif family == "global_rate":
        counts = np.ceil(proof_global_nights["expected"]).astype(int).to_numpy()
    elif family == "hotel_rate":
        counts = np.ceil(proof_hotel_nights["expected"]).astype(int).to_numpy()
    elif family == "hotel_month_lookup":
        counts = np.ceil(proof_lookup_nights["expected"]).astype(int).to_numpy()
    elif family == "model_expected":
        counts = np.ceil(proof_nights["expected"]).astype(int).to_numpy()
    elif family == "model_buffer":
        counts = np.ceil(proof_nights["expected"] + rule["multiplier"] * proof_nights["spread"]).astype(int).to_numpy()
    elif family == "model_margin":
        counts = np.maximum(np.ceil(proof_nights["expected"]).astype(int).to_numpy() - rule["margin"], 0)
    elif family == "flat_rule_margin":
        counts = np.maximum(np.ceil(proof_hotel_nights["expected"]).astype(int).to_numpy() - rule["margin"], 0)
    else:
        raise ValueError("Unsupported frozen policy")
    assert len(counts) == len(proof_nights) and np.all(counts >= 0)
    return counts


proof_oversell = oversell_for(FROZEN_RULE)
print("Applied frozen policy:", FROZEN_POLICY_NAME)

Applied frozen policy: flat_rule_margin_17


## 11. Score the frozen policy using the client contract

This is the primary Proof-period score. The supplied contract computes total cost, simulated empty rooms, simulated walks, walks per 1,000 actual arrivals and whether the guardrail was met.

In [13]:
proof_score = contract.night_cost(
    proof_nights,
    proof_oversell
)

proof_score_table = pd.DataFrame([{
    "model": FROZEN_MODEL_NAME,
    "model_settings": str(FROZEN_MODEL_SETTINGS),
    "policy": FROZEN_POLICY_NAME,
    **proof_score
}])

display(proof_score_table)

,model,model_settings,policy,cost_eur,empty_rooms,walked_guests,walks_per_1000,ceiling_met,nights
0,logistic_regression,{'C': 0.1},flat_rule_margin_17,3935875.6,8186,155,6.21,True,486


In [14]:
# The declared benchmarks, scored on the same Proof hotel-nights (project rule 5), beside their Choose figures.
benchmark_rows = [{"role": "frozen selected policy", "policy": FROZEN_POLICY_NAME, "period": "Choose",
                   **frozen["choose_scores"]["selected"]},
                  {"role": "frozen selected policy", "policy": FROZEN_POLICY_NAME, "period": "Proof", **proof_score}]
for role, rule in BENCHMARKS.items():
    if rule is None:
        continue
    benchmark_rows.append({"role": role, "policy": rule["policy"], "period": "Proof",
                           **contract.night_cost(proof_nights, oversell_for(rule))})
benchmark_table = pd.DataFrame(benchmark_rows)
flat_cost = benchmark_table.query("role == 'flat rule (contract baseline)'")["cost_eur"]
if len(flat_cost):
    proof_rows = benchmark_table["period"] == "Proof"
    benchmark_table.loc[proof_rows, "vs flat rule (EUR)"] = benchmark_table.loc[proof_rows, "cost_eur"] - flat_cost.iloc[0]
display(benchmark_table.drop(columns=["policy"], errors="ignore").set_index(["role", "period"]).join(
    benchmark_table.set_index(["role", "period"])[["policy"]]))

cost_eur  empty_rooms  \
role                              period                            
frozen selected policy            Choose  2414409.40         5532   
                                  Proof   3935875.60         8186   
flat rule (contract baseline)     Proof   3935875.60         8186   
best model-based policy on Choose Proof   4172660.07         8615   

                                          walked_guests  walks_per_1000  \
role                              period                                  
frozen selected policy            Choose            169            9.06   
                                  Proof             155            6.21   
flat rule (contract baseline)     Proof             155            6.21   
best model-based policy on Choose Proof             277           11.11   

                                          ceiling_met  nights  \
role                              period                        
frozen selected policy            Choose         True     NaN   
                                  Proof          True   486.0   
flat rule (contract baseline)     Proof          True   486.0   
best model-based policy on Choose Proof         False   486.0   

                                          vs flat rule (EUR)  \
role                              period                       
frozen selected policy            Choose                 NaN   
                                  Proof                 0.00   
flat rule (contract baseline)     Proof                 0.00   
best model-based policy on Choose Proof            236784.47   

                                                       policy  
role                              period                       
frozen selected policy            Choose  flat_rule_margin_17  
                                  Proof   flat_rule_margin_17  
flat rule (contract baseline)     Proof   flat_rule_margin_17  
best model-based policy on Choose Proof       model_margin_18

## 12. Report Proof results separately by hotel

This is a diagnostic view. The supplied contract's group-level walk ceiling remains the formal criterion unless the client specifies separate property-level limits.

In [15]:
hotel_proof_rows = []

for hotel_name in proof_nights.index.get_level_values("hotel").unique():
    hotel_mask = proof_nights.index.get_level_values("hotel") == hotel_name
    hotel_nights = proof_nights.loc[hotel_mask]
    hotel_oversell = proof_oversell[hotel_mask]

    hotel_score = contract.night_cost(
        hotel_nights,
        hotel_oversell
    )

    hotel_proof_rows.append({
        "hotel": hotel_name,
        **hotel_score
    })

hotel_proof_scores = pd.DataFrame(hotel_proof_rows)
display(hotel_proof_scores)

,hotel,cost_eur,empty_rooms,walked_guests,walks_per_1000,ceiling_met,nights
0,City Hotel,1640856.89,4383,146,9.23,True,243
1,Resort Hotel,2295018.71,3803,9,0.99,True,243


## 13. Inspect the nightly Proof outcomes

This table helps explain where the policy generated simulated empty rooms or walks. Actual cancellation counts are included for retrospective evaluation only; they would not be known when making a live future-night decision.

In [16]:
proof_night_detail = proof_nights.copy()
proof_night_detail["oversell_count"] = proof_oversell
proof_night_detail["empty_rooms_simulated"] = (
    proof_night_detail["cancelled"] - proof_night_detail["oversell_count"]
).clip(lower=0)
proof_night_detail["walks_simulated"] = (
    proof_night_detail["oversell_count"] - proof_night_detail["cancelled"]
).clip(lower=0)

display(proof_night_detail.head(25))

book  cancelled    expected    spread  stay_value  \
hotel      arrival_date                                                      
City Hotel 2017-01-01     103         39   62.331381  4.301767  258.124369   
           2017-01-02     124         48   45.474639  5.288324  359.438548   
           2017-01-03      95         36   31.411214  4.473005  330.703263   
           2017-01-04      61         24   19.267803  3.528850  282.513607   
           2017-01-05      65         34   26.900409  3.751709  335.478769   
           2017-01-06      69         34   26.177776  3.892776  292.839710   
           2017-01-07      45          9   17.772925  3.061325  458.025333   
           2017-01-08      81         45   47.321392  3.825916  276.043951   
           2017-01-09      45         10   14.224136  2.997804  303.729111   
           2017-01-10      64         13   24.994084  3.489179  267.964688   
           2017-01-11      37          7   12.303747  2.804240  256.715676   
           2017-01-12      96         42   44.393916  4.462085  202.238021   
           2017-01-13      70         30   22.340510  3.801896  225.048714   
           2017-01-14     169        131   46.731449  5.708433  166.116982   
           2017-01-15      49         18   21.441239  3.157168  643.603878   
           2017-01-16      61         21   20.965700  3.522012  356.618361   
           2017-01-17      31          5   11.640228  2.282022  248.835484   
           2017-01-18     190         83  122.593967  5.612537  210.716053   
           2017-01-19      71         28   30.465546  3.785027  266.083380   
           2017-01-20      60         23   20.173511  3.584281  301.815500   
           2017-01-21      69         35   23.139672  3.784370  376.416232   
           2017-01-22      65         16   20.238964  3.642439  318.232308   
           2017-01-23     140         98   84.333882  4.939997  274.268714   
           2017-01-24      52         18   13.774647  3.059992  321.765192   
           2017-01-25     150         97   94.799486  5.112996  227.286200   

                         oversell_count  empty_rooms_simulated  \
hotel      arrival_date                                          
City Hotel 2017-01-01                26                     13   
           2017-01-02                35                     13   
           2017-01-03                23                     13   
           2017-01-04                 9                     15   
           2017-01-05                11                     23   
           2017-01-06                12                     22   
           2017-01-07                 2                      7   
           2017-01-08                17                     28   
           2017-01-09                 2                      8   
           2017-01-10                10                      3   
           2017-01-11                 0                      7   
           2017-01-12                23                     19   
           2017-01-13                13                     17   
           2017-01-14                54                     77   
           2017-01-15                 4                     14   
           2017-01-16                 9                     12   
           2017-01-17                 0                      5   
           2017-01-18                63                     20   
           2017-01-19                13                     15   
           2017-01-20                 8                     15   
           2017-01-21                12                     23   
           2017-01-22                11                      5   
           2017-01-23                42                     56   
           2017-01-24                 5                     13   
           2017-01-25                46                     51   

                         walks_simulated  
hotel      arrival_date                   
City Hotel 2017-01-01                  0  
      

## 14. Save the frozen evaluation evidence

Save the frozen configuration, group score, hotel-level scores and nightly outcomes. Do not modify the configuration after inspecting these files.

In [17]:
evidence_folder = Path("evidence/proof_evaluation")
evidence_folder.mkdir(parents=True, exist_ok=True)

proof_score_table.to_csv(
    evidence_folder / "proof_group_score.csv",
    index=False
)
hotel_proof_scores.to_csv(
    evidence_folder / "proof_hotel_scores.csv",
    index=False
)
proof_night_detail.to_csv(
    evidence_folder / "proof_nightly_results.csv"
)

frozen_record = pd.DataFrame([{
    "model": FROZEN_MODEL_NAME,
    "model_settings": str(FROZEN_MODEL_SETTINGS),
    "policy": FROZEN_POLICY_NAME,
    "buffer_multiplier": (
        FROZEN_BUFFER_MULTIPLIER
        if FROZEN_POLICY_NAME.startswith("model_buffer_")
        else None
    ),
    "proof_used_for_tuning": False,
    "historically_blind_test": False
}])

frozen_record.to_csv(
    evidence_folder / "frozen_configuration.csv",
    index=False
)

print("Saved Proof evaluation evidence to:", evidence_folder)

Saved Proof evaluation evidence to: evidence\proof_evaluation


In [18]:
# Also keep the benchmark comparison and an exact copy of the freeze this run used.
benchmark_table.to_csv(evidence_folder / "proof_benchmark_comparison.csv", index=False)
(evidence_folder / "frozen_configuration_used.json").write_text(FREEZE_PATH.read_text(encoding="utf-8"), encoding="utf-8")
print("Saved benchmark comparison and the frozen configuration used.")

Saved benchmark comparison and the frozen configuration used.


## 15. Final interpretation

Report the Proof score exactly as produced by the supplied contract. Do not claim that simulated walks are actual guest walks or that booking rows establish physical room occupancy.

Discuss:
- Did the frozen policy meet the walk ceiling?
- What was its total contract cost?
- How many simulated empty room-equivalents and walks did it produce?
- Did the two hotels behave differently?
- How does Proof compare with Choose, while acknowledging the historical retrospective nature of this evaluation?
- What limitations prevent this from being a live hotel-capacity recommendation?

### Non-negotiable limitations

- The Proof period was previously inspected in earlier project work; this is not a historically blind test.
- The dataset contains no actual room capacity, rooms-per-booking, room inventory snapshots or verified actual walk records.
- The contract treats booking rows as room-equivalents for simulation.
- Feature timing has not been verified against an original booking-time PMS snapshot.
- The model probabilities follow the method frozen by Notebook 11 (calibrated or not, as Notebook 09's pre-declared rule decided).
- No changes are permitted after viewing Proof results. Any new model or policy requires a newly defined evaluation design and a separate untouched period.

### Questions

1. Why must the model and policy be frozen before evaluating Proof?
2. Why is this evaluation retrospective rather than historically blind?
3. What exactly does the contract's walked-guest count represent?
4. Why can we not claim that the model knows actual hotel capacity?
5. What evidence would be required before using this system for live overbooking decisions?

**Next notebook:** `13_ai_benchmark.ipynb`, where we document an AI-assisted benchmark and compare it with the selected analytical approach.